Name: Dev Patel 

Course: DS4400 Data Mining and Machine Learning 1

Prof: Silvio Amir

University: Northeastern University

Problem 4: Naive Bayes Classifier

1. Implement Naive Bayes from scratch. Compute prior probabilities and conditional probabilities with Laplace smoothing.
2. Estimate class probabilities for each test point.
3. Report accuracy, precision, recall, F1 score.
4. Compare with sklearn CategoricalNB package implementation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import LabelEncoder

In [2]:
# Load Mushroom dataset
# Columns: class (e=edible, p=poisonous), then 22 categorical features
mushroom_cols = [
    'class', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor',
    'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color',
    'stalk-shape', 'stalk-root', 'stalk-surface-above-ring',
    'stalk-surface-below-ring', 'stalk-color-above-ring',
    'stalk-color-below-ring', 'veil-type', 'veil-color',
    'ring-number', 'ring-type', 'spore-print-color',
    'population', 'habitat'
]

df = pd.read_csv('mushroom.data', header=None, names=mushroom_cols)
print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df['class'].value_counts())
print("\nFirst few rows:")
df.head()

Dataset shape: (8124, 23)

Class distribution:
class
e    4208
p    3916
Name: count, dtype: int64

First few rows:


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [3]:
# Encode all categorical columns as integers for both custom and sklearn implementations
label_encoders = {}
df_encoded = df.copy()

for col in df.columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Separate features and target
X = df_encoded.drop('class', axis=1)
y = df_encoded['class']  # 0 = edible, 1 = poisonous

# Map class labels for readability
class_names = label_encoders['class'].classes_
print(f"Class encoding: {dict(zip(class_names, label_encoders['class'].transform(class_names)))}")

# Split 75/25
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts()}")

Class encoding: {'e': 0, 'p': 1}

X_train: (6093, 22), X_test: (2031, 22)
Train class distribution:
class
0    3168
1    2925
Name: count, dtype: int64


### Part 1: Custom Naive Bayes Implementation with Laplace Smoothing

In [4]:
class NaiveBayesCategorical:
    """
    Naive Bayes classifier for categorical features with Laplace smoothing.
    
    Stores:
    - Prior probabilities P(Y=c) for each class c
    - Conditional probabilities P(X_i = x | Y = c) for each feature i, value x, class c
    """
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.n_classes = len(self.classes)
        n_samples = len(y)
        
        # Prior probabilities: P(Y = c)
        self.priors = {}
        for c in self.classes:
            self.priors[c] = np.sum(y == c) / n_samples
        
        print("Prior Probabilities:")
        for c in self.classes:
            print(f"  P(Y={c}) = {self.priors[c]:.4f}")
        
        # Conditional probabilities with Laplace smoothing
        # P(X_i = x | Y = c) = (count(X_i=x, Y=c) + 1) / (count(Y=c) + |V_i|)
        # where |V_i| is the number of unique values for feature i
        self.conditionals = {}
        feature_names = X.columns if hasattr(X, 'columns') else range(X.shape[1])
        X_arr = X.values if hasattr(X, 'values') else X
        y_arr = y.values if hasattr(y, 'values') else y
        
        for i, feat_name in enumerate(feature_names):
            self.conditionals[feat_name] = {}
            unique_vals = np.unique(X_arr[:, i])
            n_unique = len(unique_vals)
            
            for c in self.classes:
                class_mask = (y_arr == c)
                class_count = np.sum(class_mask)
                X_class = X_arr[class_mask, i]
                
                self.conditionals[feat_name][c] = {}
                for val in unique_vals:
                    # Laplace smoothing
                    count = np.sum(X_class == val) + 1
                    prob = count / (class_count + n_unique)
                    self.conditionals[feat_name][c][val] = prob
                
                # Store default probability for unseen values
                self.conditionals[feat_name][c]['_default'] = 1 / (class_count + n_unique)
        
        self.feature_names = list(feature_names)
        return self
    
    def predict_proba(self, X):
        X_arr = X.values if hasattr(X, 'values') else X
        n_samples = X_arr.shape[0]
        proba = np.zeros((n_samples, self.n_classes))
        
        for idx in range(n_samples):
            for j, c in enumerate(self.classes):
                # Start with log prior
                log_prob = np.log(self.priors[c])
                
                for i, feat_name in enumerate(self.feature_names):
                    val = X_arr[idx, i]
                    if val in self.conditionals[feat_name][c]:
                        log_prob += np.log(self.conditionals[feat_name][c][val])
                    else:
                        log_prob += np.log(self.conditionals[feat_name][c]['_default'])
                
                proba[idx, j] = log_prob
        
        # Convert log probabilities to probabilities using log-sum-exp trick
        max_log = np.max(proba, axis=1, keepdims=True)
        proba = np.exp(proba - max_log)
        proba = proba / np.sum(proba, axis=1, keepdims=True)
        
        return proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes[np.argmax(proba, axis=1)]

# Train custom Naive Bayes
nb_custom = NaiveBayesCategorical()
nb_custom.fit(X_train, y_train.values)

Prior Probabilities:
  P(Y=0) = 0.5199
  P(Y=1) = 0.4801


In [5]:
# Print some example conditional probabilities
print("Example conditional probabilities for feature 'odor':")
odor_feat = 'odor'
for c in nb_custom.classes:
    class_label = class_names[c]
    print(f"\n  P(odor = x | Y = {class_label}):")
    for val, prob in nb_custom.conditionals[odor_feat][c].items():
        if val != '_default':
            odor_label = label_encoders[odor_feat].classes_[val] if isinstance(val, (int, np.integer)) else val
            print(f"    P(odor={odor_label}) = {prob:.4f}")

Example conditional probabilities for feature 'odor':

  P(odor = x | Y = e):
    P(odor=a) = 0.0938
    P(odor=c) = 0.0003
    P(odor=f) = 0.0003
    P(odor=l) = 0.0913
    P(odor=m) = 0.0003
    P(odor=n) = 0.8130
    P(odor=p) = 0.0003
    P(odor=s) = 0.0003
    P(odor=y) = 0.0003

  P(odor = x | Y = p):
    P(odor=a) = 0.0003
    P(odor=c) = 0.0511
    P(odor=f) = 0.5508
    P(odor=l) = 0.0003
    P(odor=m) = 0.0102
    P(odor=n) = 0.0310
    P(odor=p) = 0.0617
    P(odor=s) = 0.1506
    P(odor=y) = 0.1438


### Part 2: Predict Class Probabilities on Test Set

In [6]:
# Predict probabilities for each test point
test_proba = nb_custom.predict_proba(X_test)
y_pred_custom = nb_custom.predict(X_test)

# Display first 10 predictions with probabilities
print("First 10 test predictions:")
print(f"{'Index':<8} {'P(Edible)':>12} {'P(Poisonous)':>14} {'Predicted':>12} {'Actual':>10}")
print("-" * 58)
for i in range(10):
    pred_label = class_names[y_pred_custom[i]]
    actual_label = class_names[y_test.values[i]]
    print(f"{i:<8} {test_proba[i, 0]:>12.6f} {test_proba[i, 1]:>14.6f} {pred_label:>12} {actual_label:>10}")

First 10 test predictions:
Index       P(Edible)   P(Poisonous)    Predicted     Actual
----------------------------------------------------------
0            0.999993       0.000007            e          e
1            0.000000       1.000000            p          p
2            0.000000       1.000000            p          p
3            1.000000       0.000000            e          e
4            0.000000       1.000000            p          p
5            0.000000       1.000000            p          p
6            0.001675       0.998325            p          p
7            0.000000       1.000000            p          p
8            1.000000       0.000000            e          e
9            0.999999       0.000001            e          e


### Part 3: Metrics for Custom Naive Bayes

In [7]:
# Compute metrics for custom implementation
custom_acc = accuracy_score(y_test, y_pred_custom)
custom_prec = precision_score(y_test, y_pred_custom)
custom_rec = recall_score(y_test, y_pred_custom)
custom_f1 = f1_score(y_test, y_pred_custom)

print("Custom Naive Bayes - Test Set Metrics:")
print(f"  Accuracy:  {custom_acc:.4f}")
print(f"  Precision: {custom_prec:.4f}")
print(f"  Recall:    {custom_rec:.4f}")
print(f"  F1 Score:  {custom_f1:.4f}")

Custom Naive Bayes - Test Set Metrics:
  Accuracy:  0.9488
  Precision: 0.9901
  Recall:    0.9041
  F1 Score:  0.9451


### Part 4: Comparison with sklearn CategoricalNB

In [8]:
# Train sklearn CategoricalNB (alpha=1 for Laplace smoothing)
nb_sklearn = CategoricalNB(alpha=1.0)
nb_sklearn.fit(X_train, y_train)
y_pred_sklearn = nb_sklearn.predict(X_test)

sklearn_acc = accuracy_score(y_test, y_pred_sklearn)
sklearn_prec = precision_score(y_test, y_pred_sklearn)
sklearn_rec = recall_score(y_test, y_pred_sklearn)
sklearn_f1 = f1_score(y_test, y_pred_sklearn)

print("sklearn CategoricalNB - Test Set Metrics:")
print(f"  Accuracy:  {sklearn_acc:.4f}")
print(f"  Precision: {sklearn_prec:.4f}")
print(f"  Recall:    {sklearn_rec:.4f}")
print(f"  F1 Score:  {sklearn_f1:.4f}")

# Side-by-side comparison
print("\n\n=== Comparison: Custom vs sklearn Naive Bayes ===")
print(f"{'Metric':<12} {'Custom':>10} {'sklearn':>10} {'Difference':>12}")
print("-" * 46)
print(f"{'Accuracy':<12} {custom_acc:>10.4f} {sklearn_acc:>10.4f} {abs(custom_acc - sklearn_acc):>12.4f}")
print(f"{'Precision':<12} {custom_prec:>10.4f} {sklearn_prec:>10.4f} {abs(custom_prec - sklearn_prec):>12.4f}")
print(f"{'Recall':<12} {custom_rec:>10.4f} {sklearn_rec:>10.4f} {abs(custom_rec - sklearn_rec):>12.4f}")
print(f"{'F1 Score':<12} {custom_f1:>10.4f} {sklearn_f1:>10.4f} {abs(custom_f1 - sklearn_f1):>12.4f}")

sklearn CategoricalNB - Test Set Metrics:
  Accuracy:  0.9488
  Precision: 0.9901
  Recall:    0.9041
  F1 Score:  0.9451


=== Comparison: Custom vs sklearn Naive Bayes ===
Metric           Custom    sklearn   Difference
----------------------------------------------
Accuracy         0.9488     0.9488       0.0000
Precision        0.9901     0.9901       0.0000
Recall           0.9041     0.9041       0.0000
F1 Score         0.9451     0.9451       0.0000


**Observations - Custom vs sklearn Naive Bayes:**

- The results from the custom implementation and sklearn's CategoricalNB should be very similar (or identical), since both use the same Naive Bayes formulation with Laplace smoothing (alpha=1).
- Any minor differences may arise from numerical precision (e.g., log-space vs direct multiplication) or how each implementation handles unseen feature values in the test set.
- The Naive Bayes classifier performs well on the Mushroom dataset because the conditional independence assumption holds reasonably well for this data — many features (like odor, spore print color) are highly discriminative on their own.
- The high accuracy and F1 scores demonstrate that Naive Bayes is effective for categorical classification tasks, especially when features are informative and relatively independent.